# Phase 3 — Source-Held-Out Probes

**The question.** Does the model read clauses, or does it recognise datasets? The fused
corpus stitches three annotation projects together, each with its own drafting register,
clause segmentation and label vocabulary. A model that has learned "this looks like a
CLAUDETTE row, and CLAUDETTE rows about arbitration are usually harmful" would score well
on a random split while having learned very little about arbitration.

The existing `notebooks/model_finetuning/lawgic_classifier_probe.ipynb` already showed
that source identity is *linearly decodable* from the fine-tuned encoder. That is
necessary but not sufficient evidence: an encoder can carry source information without the
heads depending on it. This notebook tests the stronger claim directly — **remove a source
from training entirely, then evaluate only on that source's rows.**

## Two probes, and why not three

| Probe | Held out | Corpus rows carrying that source |
| --- | --- | --- |
| A | CLAUDETTE | 3,182 wide rows (3,721 long-format annotation rows) |
| B | 100 ToS | 1,460 wide rows (2,048 long-format annotation rows) |
| — | ~~ToS;DR~~ | **deliberately not run** |

The row counts differ between the long and wide formats because the wide corpus is one row
per unique clause: a clause annotated with several topics by the same source collapses into
one row with several active topic cells. The holdout operates on wide rows, so those are
the numbers reported.

**Why there is no ToS;DR holdout.** ToS;DR supplies the majority of the corpus rows — the majority of the corpus and training rows once the split is applied. Removing it would
leave very few training clauses. A score collapse under that condition is uninterpretable:
it would be perfectly consistent with "the model only recognised ToS;DR" *and* with "no
model learns 44-way multi-label legal topic detection from 2,500 examples". The probe
would be confounded with data starvation and would answer neither question. The two
smaller sources can be removed while leaving the training regime broadly intact, which is
what makes their results readable.

## Protocol

Legal-BERT, seed 42, dual-head, identical to Phase 2 in every other respect — same
persisted seed-42 split, same hyperparameters, same losses, same early stopping, same
degenerate-model assertion. The holdout removes the source's rows from **train and
validation** and restricts the **test** set to exactly those rows.


## Masking: score only what the held-out source actually supervised

This is the part that decides whether the probe means anything, and the first pass got the
direction right and the width wrong.

The supervision mask is source-aware: a row annotated by CLAUDETTE has observed cells only
for the topics CLAUDETTE's label vocabulary covers; every other cell is *unknown* and
contributes zero loss. If a held-out CLAUDETTE row is scored across all predicted topics, most of
the score comes from cells CLAUDETTE never labelled — the model is being graded against
the shape of the mask, not against comprehension of the clause. So the surface must be
restricted.

**But it must not be restricted to the cells the source marked positive.**
`source_supervision_mask()` in `scripts/lawgic_train_matrix.py` rebuilds the mask from
`native_annotations`, and `native_annotations` records only *asserted* annotations — that
is, only positives. The resulting surface therefore contains **zero observed negatives**
(CLAUDETTE: 375 cells, 375 positive; 100 ToS: 197 cells, 197 positive). False positives
become structurally impossible, micro-precision comes out at exactly 1.000 in every
condition including in-distribution, and per topic F1 collapses to `2r/(1+r)` — a monotone
function of recall alone. A model predicting every topic present for every clause would
score a perfect 1.000 on that surface. It is the positive-only-corpus degeneracy this
project already fixed once at the corpus level, reappearing one layer up in the evaluation.

So two surfaces are reported below:

| Surface | Mask | Observed negatives | What it measures |
| --- | --- | --- | --- |
| **asserted** | `tm.source_supervision_mask()` | none | recall on the topics the source marked; the as-first-published figures |
| **corpus** | the row's own `label_mask` | yes | precision *and* recall; the honest F1 |

The corpus mask is the fusion pipeline's own source-aware mask, built by applying the
mapping policy, so for a row annotated by one source it *is* that source's supervision.
`source_mappings` in `lawgic_topics*.json` cannot be used instead: it lists every topic a
source's raw labels touch, including the fine subtypes the mapping policy deliberately
refuses to assert (CLAUDETTE's `ltd` maps to `limitation_of_liability` alone, not to
`liability_cap` and `warranty_disclaimer`), so masking on it would manufacture false
negatives — the exact error the mapping layer exists to avoid.

Rows carrying more than one source would mix their masks, so `PURE_SOURCE_ONLY` restricts
the corpus surface to rows annotated by the held-out source alone. That is 96% of them
(cross-source overlap is 0.4% of the corpus).

The in-distribution baseline is computed on **exactly the same rows and the same cell
mask**, from the Phase 2 legal-bert/seed-42 run's stored logits. Without that restriction
the "retained ratio" would compare two different denominators and would be meaningless.


In [5]:
import os
import sys
from pathlib import Path

# ── Corpus version: set BEFORE importing lawgic_eval_core ─────────────────────
os.environ["LAWGIC_CORPUS_VERSION"] = "v2"


def find_project_root(start: Path) -> Path:
    for sentinel in [
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv",
        "generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv",
    ]:
        for candidate in (start, *start.parents):
            if (candidate / sentinel).exists():
                return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS

HOLDOUT_SOURCES = ["claudette", "100_tos"]
BASELINE_RUN = tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual").run_id

for source in HOLDOUT_SOURCES + ["tos_dr"]:
    corpus_rows = int(tm.source_row_mask(corpus, source).sum())
    train_rows = int(tm.source_row_mask(frames["train"], source).sum())
    test_rows = int(tm.source_row_mask(frames["test"], source).sum())
    print(f"{source:>10}: corpus {corpus_rows:>6,} | train {train_rows:>6,} "
          f"({train_rows / len(frames['train']):.1%}) | test {test_rows:>5,}")

print(f"\nCorpus version: {core._CORPUS_VERSION}")
print(f"Topics: {core.NUM_LAWGIC_TOPICS}")
print(f"Runs dir: {tm.RUNS_DIR}")
print(f"Phase 2 baseline run required: {BASELINE_RUN}")


 claudette: corpus  3,182 | train  2,535 (11.9%) | test   331
   100_tos: corpus  1,535 | train  1,239 (5.8%) | test   149
    tos_dr: corpus 21,949 | train 17,554 (82.6%) | test 2,190

Corpus version: v2
Topics: 42
Runs dir: C:\Users\Enrique\Coding Projects\Thesis\lawgic\saved_models\multiseed_encoder_runs_v2
Phase 2 baseline run required: legal-bert-base-uncased__seed42__dual


## Prerequisites

1. **Phase 2 must have run.** This notebook reads
   `generated_files/lawgic_taxonomy/runs/legal-bert-base-uncased__seed42__dual/test_logits.npz`
   for the in-distribution baseline. Without it there is nothing to compare against.
2. **No GPU needed.** Both probes are already trained and their test logits are persisted, so
   every cell below re-scores stored arrays and runs on a laptop in seconds. The training call
   is commented out; uncomment it only if a probe directory is missing.
3. No downloads or credentials.

In [6]:
baseline_path = tm.RUNS_DIR / BASELINE_RUN / "test_logits.npz"
if not baseline_path.exists():
    raise FileNotFoundError(
        f"{baseline_path} missing. Run 02_multiseed_encoder_runs.ipynb (at least the "
        f"legal-bert/seed42/dual config) before this notebook."
    )
print(f"Baseline logits found: {baseline_path}")

Baseline logits found: C:\Users\Enrique\Coding Projects\Thesis\lawgic\saved_models\multiseed_encoder_runs_v2\legal-bert-base-uncased__seed42__dual\test_logits.npz


## Load the two probes

Each probe retrains Legal-BERT with one source removed from train and validation
(~40-55 min per run on a CUDA GPU, much longer on CPU). Every probe persists its
metrics, per-topic table and test logits on its first run, so the cell below
loads any probe already on disk and trains only the ones that are missing.

`legal-bert-base-uncased__seed42__dual__holdout-claudette` is already trained
(53.8 min, 18 epochs, early-stopped inside the 20-epoch budget);
`…__holdout-100_tos` is trained on the first run that reaches this cell.

Once both probes exist, everything after this cell re-scores their **persisted
logits**, so the rest of the notebook runs on a laptop in seconds with no GPU.

In [7]:
probe_configs = [
    tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual", holdout_source=source)
    for source in HOLDOUT_SOURCES
]

# Each probe persists its metrics, per-topic table and test logits on the first
# run, so this cell loads any probe already on disk and trains only the ones
# that are missing. A missing probe is trained here and then behaves like the
# others — ~40-55 min per run on a CUDA GPU, much longer on CPU.
probe_records = []
for config in probe_configs:
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists():
        print(f"loaded {config.run_id}")
        probe_records.append(json.loads(target.read_text()))
        continue

    print(f"training {config.run_id} (not found on disk) ...")
    probe_records.append(tm.run_config(config))
    print(f"saved {target}")

display(pd.DataFrame(probe_records)[
    ["run_id", "train_rows", "val_rows", "test_rows", "wall_seconds", "epochs_run",
     *core.HEADLINE_METRICS]
])
print("\nNote: the metrics above are on the *asserted* surface (positives only) — that is what "
      "run_config computed at training time. The corrected figures follow below.")

loaded legal-bert-base-uncased__seed42__dual__holdout-claudette
training legal-bert-base-uncased__seed42__dual__holdout-100_tos (not found on disk) ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.700000,0.740673,0.120382,0.351924,0.272707,88478.000000,0.806220,0.795903,0.806486,2508.000000
2,0.454500,0.670594,0.465946,0.683574,0.633497,88478.000000,0.820574,0.806999,0.817587,2508.000000
3,0.420400,0.630866,0.610112,0.774007,0.754946,88478.000000,0.842504,0.833706,0.842310,2508.000000
4,0.309100,0.789678,0.667237,0.812740,0.801578,88478.000000,0.840909,0.829563,0.839304,2508.000000
5,0.196600,0.805166,0.696110,0.832784,0.826156,88478.000000,0.846890,0.837655,0.845792,2508.000000
6,0.137600,0.882010,0.718222,0.839183,0.836335,88478.000000,0.853270,0.842042,0.851224,2508.000000
7,0.102200,1.026853,0.721752,0.840726,0.838517,88478.000000,0.850478,0.839775,0.849526,2508.000000
8,0.071500,1.000170,0.729251,0.845624,0.844371,88478.000000,0.857257,0.847571,0.856480,2508.000000
9,0.084800,1.125376,0.730762,0.841770,0.839590,88478.000000,0.854466,0.844369,0.853538,2508.000000
10,0.045700,1.117480,0.729086,0.840864,0.839652,88478.000000,0.860845,0.851276,0.860567,2508.000000


[legal-bert-base-uncased__seed42__dual__holdout-100_tos] 66.8 min | topic_macro_f1=0.5553 topic_micro_f1=0.7333 risk_accuracy=0.4832 risk_macro_f1=0.4114
saved C:\Users\Enrique\Coding Projects\Thesis\lawgic\saved_models\multiseed_encoder_runs_v2\legal-bert-base-uncased__seed42__dual__holdout-100_tos\metrics.json


,run_id,train_rows,val_rows,test_rows,wall_seconds,epochs_run,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1
0,legal-bert-base-uncased__seed42__dual__holdout...,18708,2339,331,3225.807584,18.0,0.625081,0.630237,0.507553,0.384990
1,legal-bert-base-uncased__seed42__dual__holdout...,20004,2508,149,4005.774901,20.0,0.555263,0.733333,0.483221,0.411389



Note: the metrics above are on the *asserted* surface (positives only) — that is what run_config computed at training time. The corrected figures follow below.


## In-distribution baseline on the same rows and the same cells

The Phase 2 run scored the full test split. Here it is re-scored on the subset of
test rows belonging to the held-out source, under both candidate masks — the identical
evaluation surface the probe faces, so the only difference between the two conditions is
whether the source appeared in training.

Both conditions are recomputed from persisted logits by the same code path, rather than
reading the probe's `metrics.json`, so the two masks can be applied consistently and the
`risk_majority_floor` column can be computed on the same rows. A metric without its floor
is uninterpretable, and the floor moves with the subset: predicting *neutral* everywhere
scores 0.744 on CLAUDETTE's test rows and 0.610 on 100 ToS's, against 0.469 on the full
test split.


In [8]:
SURFACES = ("corpus", "asserted")   # first one is the headline
CONDITIONS = ("in-distribution (Phase 2)", "held-out (Phase 3)")
IN_DIST, HELD_OUT = CONDITIONS

baseline = np.load(baseline_path)
base_position = {int(r): i for i, r in enumerate(baseline["row_id"])}
test_frame = frames["test"]


def micro_pr(topic_logits, labels, masks):
    """Micro precision / recall over observed cells. Catches over-prediction, which the
    asserted surface cannot: with no observed negatives, fp is 0 by construction."""
    pred = core.logits_to_predictions(topic_logits).astype(bool)
    lab, obs = labels.astype(bool), masks.astype(bool)
    tp = int((pred & lab & obs).sum())
    fp = int((pred & ~lab & obs).sum())
    fn = int((~pred & lab & obs).sum())
    return {
        "micro_precision": tp / (tp + fp) if tp + fp else 0.0,
        "micro_recall": tp / (tp + fn) if tp + fn else 0.0,
        "false_positives": fp,
    }


def risk_majority_floor(arrays):
    """Accuracy of always predicting the most frequent risk class on these rows."""
    valid = arrays["harm_masks"].astype(bool)
    counts = np.bincount(arrays["harm_labels"][valid].astype(int), minlength=core.NUM_HARM_CLASSES)
    return float(counts.max() / counts.sum())


def probe_surface(source, pure_source_only=False):
    """Aligned logits for both conditions plus both candidate masks, on one probe's rows."""
    probe = np.load(tm.RUNS_DIR / f"{BASELINE_RUN}__holdout-{source}" / "test_logits.npz")
    probe_position = {int(r): i for i, r in enumerate(probe["row_id"])}

    rows = test_frame[tm.source_row_mask(test_frame, source)].copy()
    if pure_source_only:
        rows = rows[rows["sources"].map(lambda s: list(s) == [source])].copy()

    row_ids = [int(r) for r in rows["row_id"]]
    base_idx = np.array([base_position[i] for i in row_ids])
    probe_idx = np.array([probe_position[i] for i in row_ids])

    arrays = core.label_arrays(rows)
    return {
        "source": source,
        "arrays": arrays,
        # "corpus": the fusion pipeline's own source-aware mask, which carries observed
        # negatives. "asserted": only the cells this source marked positive.
        "masks": {"corpus": arrays["label_masks"],
                  "asserted": tm.source_supervision_mask(rows, source)},
        "logits": {IN_DIST: (baseline["topic_logits"][base_idx], baseline["harm_logits"][base_idx]),
                   HELD_OUT: (probe["topic_logits"][probe_idx], probe["harm_logits"][probe_idx])},
        "n_rows": len(rows),
    }


probes = {source: probe_surface(source) for source in HOLDOUT_SOURCES}

records = []
for source, probe in probes.items():
    arrays = probe["arrays"]
    floor = risk_majority_floor(arrays)
    for surface in SURFACES:
        mask = probe["masks"][surface]
        scoped = {**arrays, "label_masks": mask}
        observed = int(mask.sum())
        positives = int((arrays["labels"] * mask).sum())
        for condition, (topic_logits, harm_logits) in probe["logits"].items():
            records.append({
                "source": source,
                "surface": surface,
                "condition": condition,
                "rows": probe["n_rows"],
                "observed_cells": observed,
                "positive_cells": positives,
                "negative_cells": observed - positives,
                **core.all_metrics(topic_logits, harm_logits, scoped),
                **micro_pr(topic_logits, arrays["labels"], mask),
                "risk_majority_floor": floor,
            })

comparison = pd.DataFrame(records).drop(columns=["topic_observed_positions", "risk_valid_rows"])
display(comparison.set_index(["source", "surface", "condition"]).round(4))

# The defect, stated as an assertion rather than a claim.
asserted = comparison[comparison["surface"] == "asserted"]
assert (asserted["negative_cells"] == 0).all(), "asserted surface unexpectedly has negatives"
assert (asserted["false_positives"] == 0).all()
print("\nasserted surface: 0 observed negatives and 0 false positives in every condition — "
      "its F1 is a recall proxy, precision is free.")

# Sensitivity check: does dropping the few multi-source rows move anything?
for source in HOLDOUT_SOURCES:
    pure = probe_surface(source, pure_source_only=True)
    scoped = {**pure["arrays"], "label_masks": pure["masks"]["corpus"]}
    pure_held = core.all_metrics(*pure["logits"][HELD_OUT], scoped)
    full = comparison[(comparison["source"] == source)
                      & (comparison["surface"] == "corpus")
                      & (comparison["condition"] == HELD_OUT)].iloc[0]
    print(f"{source:>10}: corpus-surface held-out micro-F1 "
          f"{full['topic_micro_f1']:.4f} on {full['rows']} rows | "
          f"{pure_held['topic_micro_f1']:.4f} on {pure['n_rows']} single-source rows")

rows  observed_cells  positive_cells  negative_cells  topic_macro_f1  topic_micro_f1  topic_weighted_f1  \
source    surface  condition                                                                                                                             
claudette corpus   in-distribution (Phase 2)  331.0             660             385             275          0.2942          0.8828             0.8975   
                   held-out (Phase 3)         331.0             660             385             275          0.2385          0.6375             0.5647   
          asserted in-distribution (Phase 2)  331.0             376             376               0          0.9258          0.9117             0.9096   
                   held-out (Phase 3)         331.0             376             376               0          0.6251          0.6302             0.5578   
100_tos   corpus   in-distribution (Phase 2)  149.0            4098             206            3892          0.4208          0.6556             0.6814   
                   held-out (Phase 3)         149.0            4098             206            3892          0.2268          0.3029             0.3839   
          asserted in-distribution (Phase 2)  149.0             190             190               0          0.8325          0.8690             0.8533   
                   held-out (Phase 3)         149.0             190             190               0          0.5553          0.7333             0.6419   

                                              risk_accuracy  risk_macro_f1  risk_weighted_f1  micro_precision  micro_recall  false_positives  \
source    surface  condition                                                                                                                   
claudette corpus   in-distribution (Phase 2)         0.7885         0.6831            0.7953           0.9284        0.8416               25   
                   held-out (Phase 3)                0.5076         0.3850            0.5423           0.9785        0.4727                4   
          asserted in-distribution (Phase 2)         0.7885         0.6831            0.7953           1.0000        0.8378                0   
                   held-out (Phase 3)                0.5076         0.3850            0.5423           1.0000        0.4601                0   
100_tos   corpus   in-distribution (Phase 2)         0.6510         0.5581            0.6577           0.5725        0.7670              118   
                   held-out (Phase 3)                0.4832         0.4114            0.4836           0.2013        0.6117              500   
          asserted in-distribution (Phase 2)         0.6510         0.5581            0.6577           1.0000        0.7684                0   
                   held-out (Phase 3)                0.4832         0.4114            0.4836           1.0000        0.5789                0   

                                              risk_majority_floor  
source    surface  condition                                       
claudette corpus   in-distribution (Phase 2)               0.7402  
                   held-out (Phase 3)                      0.7402  
          asserted in-distribution (Phase 2)               0.7402  
                   held-out (Phase 3)                      0.7402  
100_tos   corpus   in-distribution (Phase 2)               0.5839  
                   held-out (Phase 3)                      0.5839  
          asserted in-distribution (Phase 2)               0.5839  
                   held-out (Phase 3)                      0.5839


asserted surface: 0 observed negatives and 0 false positives in every condition — its F1 is a recall proxy, precision is free.
 claudette: corpus-surface held-out micro-F1 0.6375 on 331.0 rows | 0.6278 on 322 single-source rows
   100_tos: corpus-surface held-out micro-F1 0.3029 on 149.0 rows | 0.2714 on 139 single-source rows


## Retained-performance ratio, with a paired bootstrap on the difference

`held-out / in-distribution`, per metric. Read it as: **what fraction of its ability does the
model keep when it has never seen a single clause from this source?**

- **near 1.0** — the model generalises across annotation projects; performance is not an
  artifact of source recognition.
- **materially below 1.0** — part of the headline score depends on having seen that source's
  register during training. That is a real limitation of the fused-corpus design, not
  necessarily a modelling failure.

A ratio on its own is not evidence, so two things are attached to it. The **observed-cell
count** is reported so the ratio is never read without its denominator. And the difference
between the two conditions is **paired-bootstrapped over the same clause resamples**, because
both conditions scored identical rows in identical order — if that interval contains zero, the
drop is indistinguishable from sampling noise. It does not contain zero here.

Metrics are reported on both surfaces. Prefer the `corpus` rows: on the `asserted` surface the
precision columns are 1.000 by construction, so its `topic_macro_f1` is a recall proxy.

In [9]:
RATIO_METRICS = (*core.HEADLINE_METRICS, "micro_precision", "micro_recall")
N_RESAMPLES = 1000


def delta_ci(probe, mask, metric):
    """Paired bootstrap 95% CI on (held-out minus in-distribution) for one metric.

    Both conditions are scored on the *same* resampled clause indices, so clause
    difficulty cancels — the interval is over the difference, not over two scores.
    """
    arrays = {**probe["arrays"], "label_masks": mask}
    in_topic, in_harm = probe["logits"][IN_DIST]
    held_topic, held_harm = probe["logits"][HELD_OUT]

    def scorer(logits):
        topic_logits, harm_logits = logits

        def score(idx):
            values = core.all_metrics(topic_logits, harm_logits, arrays, idx)
            if metric in values:
                return values[metric]
            return micro_pr(topic_logits[idx], arrays["labels"][idx], mask[idx])[metric]

        return score

    held_score, in_score = scorer((held_topic, held_harm)), scorer((in_topic, in_harm))
    return core.paired_bootstrap_delta(
        lambda idx: held_score(idx) - in_score(idx),
        np.arange(probe["n_rows"]),
        n_resamples=N_RESAMPLES,
    )


ratio_rows = []
for source, probe in probes.items():
    for surface in SURFACES:
        mask = probe["masks"][surface]
        scored = {
            condition: {
                **core.all_metrics(*logits, {**probe["arrays"], "label_masks": mask}),
                **micro_pr(logits[0], probe["arrays"]["labels"], mask),
            }
            for condition, logits in probe["logits"].items()
        }
        for metric in RATIO_METRICS:
            in_value, held_value = scored[IN_DIST][metric], scored[HELD_OUT][metric]
            test = delta_ci(probe, mask, metric)
            ratio_rows.append({
                "Source": source,
                "Surface": surface,
                "Metric": metric,
                "In-distribution": in_value,
                "Held-out": held_value,
                "Retained ratio": held_value / in_value if in_value else float("nan"),
                "Delta": test["delta"],
                "CI low": test["ci_low"],
                "CI high": test["ci_high"],
                "Test rows": probe["n_rows"],
                "Observed cells": int(mask.sum()),
                "Observed negatives": int(mask.sum() - (probe["arrays"]["labels"] * mask).sum()),
            })

probe_table = pd.DataFrame(ratio_rows)
for surface in SURFACES:
    print(f"\n=== {surface} surface ===")
    display(probe_table[probe_table["Surface"] == surface].drop(columns="Surface").round(4))

core.write_outputs(
    probe_table[probe_table["Surface"] == "corpus"].drop(columns="Surface"),
    "phase3_source_holdout",
    caption=(
        "Source-held-out probes. Each probe retrains Legal-BERT (seed 42, dual-head, "
        "identical protocol) with one source removed from train and validation, then "
        "evaluates only on that source's test rows under that source's own supervision "
        "mask. The in-distribution column is the Phase 2 legal-bert/seed-42 run scored on "
        "the identical rows and cells, so the columns differ only in training experience. "
        "Confidence intervals are percentile bootstrap over 1{,}000 clause-level resamples "
        "of the paired difference. No ToS;DR probe is reported: it would remove ~83\\% of "
        "training rows, confounding source recognition with data starvation."
    ),
    label="tab:source-holdout",
)

core.write_outputs(
    probe_table[probe_table["Surface"] == "asserted"].drop(columns="Surface"),
    "phase3_source_holdout_asserted",
    caption=(
        "Source-held-out probes scored on the asserted-cells surface, which contains only "
        "the topic cells the held-out source marked positive and therefore no observed "
        "negatives. Precision is 1.000 by construction, so these F1 figures are recall "
        "proxies; they are reported for continuity with the run-time metrics in "
        "\\texttt{metrics.json}. Table \\ref{tab:source-holdout} is the headline result."
    ),
    label="tab:source-holdout-asserted",
)


=== corpus surface ===


,Source,Metric,In-distribution,Held-out,Retained ratio,Delta,CI low,CI high,Test rows,Observed cells,Observed negatives
0,claudette,topic_macro_f1,0.2942,0.2385,0.8109,-0.0556,-0.0818,-0.0375,331,660,275
1,claudette,topic_micro_f1,0.8828,0.6375,0.7221,-0.2454,-0.3060,-0.1756,331,660,275
2,claudette,risk_accuracy,0.7885,0.5076,0.6437,-0.2810,-0.3384,-0.2236,331,660,275
3,claudette,risk_macro_f1,0.6831,0.3850,0.5636,-0.2981,-0.3735,-0.2102,331,660,275
4,claudette,micro_precision,0.9284,0.9785,1.0540,0.0501,-0.0094,0.1439,331,660,275
5,claudette,micro_recall,0.8416,0.4727,0.5617,-0.3688,-0.4248,-0.3111,331,660,275
12,100_tos,topic_macro_f1,0.4208,0.2268,0.5390,-0.1940,-0.2402,-0.1408,149,4098,3892
13,100_tos,topic_micro_f1,0.6556,0.3029,0.4620,-0.3527,-0.4298,-0.2737,149,4098,3892
14,100_tos,risk_accuracy,0.6510,0.4832,0.7423,-0.1678,-0.2349,-0.1074,149,4098,3892
15,100_tos,risk_macro_f1,0.5581,0.4114,0.7371,-0.1467,-0.2081,-0.0911,149,4098,3892



=== asserted surface ===


,Source,Metric,In-distribution,Held-out,Retained ratio,Delta,CI low,CI high,Test rows,Observed cells,Observed negatives
6,claudette,topic_macro_f1,0.9258,0.6251,0.6752,-0.3007,-0.3481,-0.2533,331,376,0
7,claudette,topic_micro_f1,0.9117,0.6302,0.6913,-0.2815,-0.3319,-0.2327,331,376,0
8,claudette,risk_accuracy,0.7885,0.5076,0.6437,-0.2810,-0.3384,-0.2236,331,376,0
9,claudette,risk_macro_f1,0.6831,0.3850,0.5636,-0.2981,-0.3735,-0.2102,331,376,0
10,claudette,micro_precision,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,331,376,0
11,claudette,micro_recall,0.8378,0.4601,0.5492,-0.3777,-0.4320,-0.3202,331,376,0
18,100_tos,topic_macro_f1,0.8325,0.5553,0.6670,-0.2772,-0.3335,-0.2084,149,190,0
19,100_tos,topic_micro_f1,0.8690,0.7333,0.8438,-0.1357,-0.1925,-0.0873,149,190,0
20,100_tos,risk_accuracy,0.6510,0.4832,0.7423,-0.1678,-0.2349,-0.1074,149,190,0
21,100_tos,risk_macro_f1,0.5581,0.4114,0.7371,-0.1467,-0.2081,-0.0911,149,190,0


(WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2/phase3_source_holdout_asserted.csv'),
 WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2/phase3_source_holdout_asserted.tex'))

### Per-topic detail, joined against the supervision that survives the holdout

Which topics survive and which collapse is more informative than the aggregate — but the
aggregate cannot be decomposed without one extra column, and without that column the probe
cannot do its job.

A topic can score zero in a probe for two completely different reasons:

- **the holdout removed its supervision.** `liability_cap`, `severability`, `service_changes`,
  `price_changes` and `limitation_period` are supplied by 100 ToS *alone*, and
  `limitation_of_liability` draws 89% of its training positives from CLAUDETTE. In the probe
  condition these topics have little or no supervision left, so a zero measures
  **unlearnability**, not a failure to read. They must be excluded from any claim about source
  recognition — this is the per-topic reappearance of the same data-starvation confound that
  makes a ToS;DR probe uninterpretable.
- **the register did not transfer.** A topic that keeps 92–99% of its supervision and still
  scores zero is the real evidence of source dependence: the model learned what the training
  source's *phrasing* of the topic looks like rather than what the contractual mechanism is.

`train_positives_lost` and `train_positives_remaining` below separate the two. Read the
`observed` column first — topics with a handful of observed cells swing wildly and are
anecdote, not measurement.

In [10]:
TOPIC_IDS, _, _ = core.load_taxonomy()
AVERAGE_ROWS = ["macro avg", "weighted avg"]

# Training positives per topic per source, from the long-format evidence on train rows.
train_positives = {}
for annotations in frames["train"]["native_annotations"]:
    for annotation in annotations or []:
        key = (annotation.get("lawgic_topic_id"), annotation.get("source_dataset"))
        train_positives[key] = train_positives.get(key, 0) + 1
supervision = (
    pd.DataFrame([{"topic_id": t, "source": s, "n": n} for (t, s), n in train_positives.items()])
    .pivot_table(index="topic_id", columns="source", values="n", fill_value=0)
    .astype(int)
)

per_topic_tables = {}
for source in HOLDOUT_SOURCES:
    # Single-source rows only here. On a co-annotated row the corpus mask is a union, so
    # the other source's cells would enter this table as one- or two-clause topics and
    # read as findings about the held-out source. The aggregates above are unaffected by
    # the restriction (see the sensitivity check in the previous cell).
    probe = probe_surface(source, pure_source_only=True)
    arrays = {**probe["arrays"], "label_masks": probe["masks"]["corpus"]}
    table = core.per_topic_table(probe["logits"][HELD_OUT][0], arrays, TOPIC_IDS)
    in_dist = core.per_topic_table(probe["logits"][IN_DIST][0], arrays, TOPIC_IDS)
    table = table.merge(
        in_dist[["topic_id", "f1"]].rename(columns={"f1": "f1_in_distribution"}),
        on="topic_id", how="left",
    )

    lost = supervision.get(source, pd.Series(dtype=int))
    remaining = supervision.drop(columns=[source], errors="ignore").sum(axis=1)
    table["train_positives_lost"] = table["topic_id"].map(lost).fillna(0).astype(int)
    table["train_positives_remaining"] = table["topic_id"].map(remaining).fillna(0).astype(int)
    table["share_lost"] = table["train_positives_lost"] / (
        table["train_positives_lost"] + table["train_positives_remaining"]
    ).replace(0, np.nan)
    per_topic_tables[source] = table

    # Topics with no positive in this subset carry no information about the holdout.
    scored = table[~table["topic_id"].isin(AVERAGE_ROWS) & (table["support"] > 0)]

    print(f"\n=== {source}: {probe['n_rows']} single-source rows, topics with at least one positive ===")
    print(scored.sort_values("f1")[
        ["topic_id", "f1", "f1_in_distribution", "precision", "recall", "support", "observed",
         "train_positives_lost", "train_positives_remaining", "share_lost"]
    ].round(4).to_string(index=False))

    starved = scored[scored["train_positives_remaining"] == 0]
    survived = scored[scored["train_positives_remaining"] > 0]
    print(f"\n  macro-F1 over all {len(scored)} scored topics:        {scored['f1'].mean():.4f}"
          f"  (in-distribution {scored['f1_in_distribution'].mean():.4f})")
    print(f"  macro-F1 over the {len(survived)} keeping supervision:  {survived['f1'].mean():.4f}"
          f"  (in-distribution {survived['f1_in_distribution'].mean():.4f})")
    if len(starved):
        print(f"  excluded, no supervision left after the holdout ({len(starved)}): "
              f"{', '.join(starved['topic_id'])}")
    collapsed = survived[(survived["f1"] == 0) & (survived["train_positives_remaining"] >= 500)]
    if len(collapsed):
        print("  register transfer failures, F1 0.000 with >=500 positives retained: "
              f"{', '.join(collapsed['topic_id'])}")


=== claudette: 322 single-source rows, topics with at least one positive ===
               topic_id     f1  f1_in_distribution  precision  recall  support  observed  train_positives_lost  train_positives_remaining  share_lost
        contract_by_use 0.0465              0.8333        1.0  0.0238       42        42                   291                       2698      0.0974
  privacy_incorporation 0.1333              0.9231        1.0  0.0714       14        14                    86                       1973      0.0418
       contract_changes 0.2667              0.8696        1.0  0.1538       39        39                   412                        827      0.3325
limitation_of_liability 0.3846              0.8830        1.0  0.2381      105       105                   861                        102      0.8941
    class_action_waiver 0.8750              0.9714        1.0  0.7778       18        18                   129                        330      0.2810
  mandatory_arbitratio